# Imports & Initializations

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
import random

DF_MESTA = pd.read_csv(
    'data/slovensko.csv',
    sep=','
)
M = len(DF_MESTA)
SUCET_POPULACIE = DF_MESTA['Population'].sum()
DF_MESTA['Weight'] = DF_MESTA['Population'] / SUCET_POPULACIE

DF_MESTA['geometry'] = [Point(xy) for xy in zip(DF_MESTA['Longitude'], DF_MESTA['Latitude'])]

DF_MESTA = gpd.GeoDataFrame(DF_MESTA, geometry='geometry', crs='EPSG:4326')
DF_MESTA = DF_MESTA.to_crs(epsg=5514)
# DF_MESTA['Points'] = DF_MESTA.geometry.apply(lambda p: [p.x, p.y]).to_numpy()
COORDS = np.column_stack([
    DF_MESTA.geometry.x,
    DF_MESTA.geometry.y,
])


WEIGHTS = DF_MESTA['Weight'].to_numpy()[:, None]

def plot_slovakia(point=None, assignments=None, cross_color='green'):
    points = point
    
    slovakia = gpd.read_file(
        "data/ne_10m_admin_0_countries.zip"
    )

    slovakia = slovakia[slovakia["ISO_A3"] == "SVK"]
    slovakia = slovakia.to_crs(epsg=5514)


    fig, ax = plt.subplots(figsize=(10,8))

    slovakia.plot(
        ax=ax,
        color='whitesmoke',
        edgecolor='black'
    )

    if assignments is None:
        DF_MESTA.plot(
            ax=ax,
            color='red',
            markersize=np.sqrt(DF_MESTA['Population']) / 5,
            alpha=0.7
        )
    else:
        DF_MESTA.assign(assignments=assignments).plot(
            ax=ax,
            column='assignments',
            markersize=np.sqrt(DF_MESTA['Population']) / 5,
            alpha=0.7
        )

    if isinstance(points, np.ndarray) and len(points.shape) == 2:
        ax.scatter(points[:, 0], points[:, 1], s=160, color=cross_color, marker="x", zorder=5)
    elif points is not None:
        ax.scatter(points[0], points[1], s=160, color=cross_color, marker="x", zorder=5)

    for x,y,city in zip(DF_MESTA.geometry.x, DF_MESTA.geometry.y, DF_MESTA['City']):
        ax.text(x,y,city, fontsize=8, ha='left', va='bottom')

    ax.set_axis_off()

    plt.tight_layout()
    plt.show()
    

In [ ]:
# print(world.columns)
# print(world[["ADMIN", "NAME", "SOVEREIGNT"]].head())
# print(world[world.astype(str).apply(lambda row: row.str.contains("Slovak", case=False).any(), axis=1)])
# minx, miny, maxx, maxy = slovakia.total_bounds
# print(minx, miny, maxx, maxy)

# Uloha a)



Mame funkciu
$$
f(x) = \sum_{i=1}^m w_i \|x-a^i\|_2
$$

kde $m$ je pocet miest, $w_i$ je vaha pridelena $i$-temu mestu a $a^i$ su 2d suradnice $i$-teho miesta 
***
Oznacme si
$$
\begin{aligned}
u_i &= x_1 - a_1^i \\
v_i &= x_2 - a_2^i \\
d_i(x) &= \sqrt{u_i^2 + v_i^2} = \|x-a^i\|_2
\end{aligned}
$$

Gradient tejto funkcie
$$

\nabla f(x) = 
\begin{pmatrix}
\displaystyle \sum_{i=1}^m w_i \frac{u_i}{d_i(x)} \\
\displaystyle \sum_{i=1}^m w_i \frac{v_i}{d_i(x)}
\end{pmatrix}
$$
V kompaktnej forme
$$
\nabla f(x) = \displaystyle \sum_{i=1}^m w_i \frac{x-a^i}{\|x-a^i\|_2}
$$

Hesseho matica
$$
\nabla^2 f(x) = 
\displaystyle \sum_{i=1}^m w_i
\begin{pmatrix}
\frac{1}{d_i} - \frac{u_i^2}{d_i^3} & -\frac{u_iv_i}{d_i^3} \\
-\frac{u_iv_i}{d_i^3} & \frac{1}{d_i} - \frac{v_i^2}{d_i^3}
\end{pmatrix}
$$

V kompaktnej forme:
$$
\nabla^2 f(x) = \displaystyle \sum_{i=1}^m w_i \left[ \frac{1}{\|x-a^i\|_2}I - \frac{(x-a^i)(x-a^i)^\top}{\|x-a^i\|^3_2} \right]
$$

# Uloha b)

**Globalne minimum existuje**: Majme hodnoty 
$$
a = min(a_1^i) \forall i \\
b = max(a_1^i) \forall i \\
c = min(a_2^i) \forall i \\
d = max(a_2^i) \forall i 
$$

Potom z tychto bodov vytvorme obdlznik s vrcholmi 
$$
[a,c], [a,d], [b,c], [b,d]
$$

Ak mame bod mimo tochto obdlznika, urcite to nebude minimum funkcie, lebo sa mozeme posunut, k najblizsiemu bodu na hrane tochto obdlznika, cim hodnotu funkcie znizime. Tym padom, ak globalne minimum existuje, musi sa nachadzat v tomto obdlzniku. 

Tento obdlznik je uzavrety a funkcia je spojita, preto mozeme pouzit Weierstrassovu vetu o extreme, ktora hovori, ze v tejto oblasti funkcia nadobudne minimum, a toto bude aj nase hladane globalne minimum.


**Riesenie nie je vo vseobecnosti jednoznacne**. Uvazujume priklad, v ktorom mame 2 mesta s rovnakym poctom obyvatelov. V takom pripade mame rovnaku hodnotu funkcie na celej usecke medzi tymito bodmi.

# Uloha c)

$$
\nabla f(x) = \displaystyle \sum_{i=1}^m w_i \frac{x-a^i}{\|x-a^i\|_2} = 0
$$



$$
\displaystyle \sum_{i=1}^m w_i \frac{x}{\|x-a^i\|_2} 
- \sum_{i=1}^m w_i \frac{a^i}{\|x-a^i\|_2} = 0
$$

$$
\displaystyle \sum_{i=1}^m w_i \frac{x}{\|x-a^i\|_2} = \sum_{i=1}^m w_i \frac{a^i}{\|x-a^i\|_2}
$$

$$
\left( \displaystyle \sum_{i=1}^m \frac{w_i}{\|x-a^i\|_2} \right) x = \sum_{i=1}^m w_i \frac{a^i}{\|x-a^i\|_2}
$$

$$
x = \frac{\displaystyle \sum_{i=1}^m \frac{w_i a^i}{\|x-a^i\|_2}}{\displaystyle \sum_{i=1}^m \frac{w_i}{\|x-a^i\|_2}}
$$

# Uloha d)

In [ ]:
# x0 = np.array([
#     (DF_MESTA.geometry.x * DF_MESTA['Weight']).sum(),
#     (DF_MESTA.geometry.y * DF_MESTA['Weight']).sum()
# ])

# x0_1 = (DF_MESTA.geometry.x * DF_MESTA['Weight']).sum()
# x0_2 = (DF_MESTA.geometry.y * DF_MESTA['Weight']).sum()


# xk_1 = x0_1
# xk_2 = x0_2

# while True:
#     s1 = 0
#     s2 = 0
    
#     for i in range(M):
#         mesto = DF_MESTA.iloc[i]

#         di = ( (xk_1 - DF_MESTA.geometry.x)**2 + (xk_2 - DF_MESTA.geometry.y)**2 )**0.5

#         h_1 = mesto['Weight'] * mesto.geometry.x
#         h_2 = mesto['Weight'] * mesto.geometry.y


WC = WEIGHTS * COORDS
EPS = 10**-6

x0 = np.sum(WC, axis=0)
xk = x0.copy()
i = 0

while True:
    d = np.linalg.norm(xk-COORDS, axis=1)[:, None]

    # print(WC / d)
    xkk = np.sum(WC/d, axis=0) / np.sum(WEIGHTS / d)
    i += 1

    if np.linalg.norm(xkk - xk) < EPS:
        xk = xkk
        break

    xk = xkk

    

print(f'Finished in {i} iterations, found: {xk}')
plot_slovakia(xk)

solution_d = xk

# a = np.array([
#     [2**0.5,2**0.5],
#     [5,5]
# ])
# np.linalg.norm(a, axis=1)

# Uloha e)

In [ ]:
def gradient_basic(step, eps, max_itrs):
    WC = WEIGHTS * COORDS
    x0 = np.sum(WC, axis=0)
    xk = x0.copy()
    iterations = []

    i = 0
    while True:
        iterations.append(xk)
        d = np.linalg.norm(xk - COORDS, axis=1)[:, None]
        g = np.sum(WEIGHTS * (xk - COORDS) / d, axis=0)

        if np.linalg.norm(g) < eps or i > max_itrs:
            break

        xk = xk - g*step
        # print(g)
        i += 1
    
    return xk, i, iterations

def find_maximum_converging_step(epsilon=10**-6):
    a = 10
    b = 5242880 
    MAX_ITR = 10**5

    while b-a > epsilon:
        c = (a+b)/2
        x,i,it = gradient_basic(c, 10**-6, MAX_ITR)
        # print(a,b,i)
        if i >= MAX_ITR:
            b = c
        else:
            a = c
    
    return a

x, i, trajectory_constant = gradient_basic(100_000, 10**-6, 100_000)
print(f'Finished in {i} iterations, found {x}')
plot_slovakia(x)

# step = 10
# while step < 10**7:
#     x, i, it = gradient_basic(step, 10**-6, 100_000)
#     print(f'Step {step} finished in {i} iterations, found {x}')
#     step *= 2

print(f'Maximum converging step: {find_maximum_converging_step(10**-6)}, precise to 5 decimal places')



# Uloha f)


In [ ]:
W = WEIGHTS.reshape(-1)

# ucelova funkcia
def objective(x):
    distances = np.linalg.norm(COORDS - x, axis=1)
    return np.sum(W * distances)


# gradient ucelovej funkcie
def gradient(x):
    distances = np.linalg.norm(x - COORDS, axis=1)

    distances = np.maximum(distances, 1e-12)

    g = np.sum(W[:, None] * (x - COORDS) / distances[:, None], axis=0)
    return g


# startovaci bod rovnaky ako v casti d) a e)
def starting_point():
    return np.sum(W[:, None] * COORDS, axis=0)


# f1) gradientna metoda s backtrackingom
def gradient_backtracking(eps=1e-6, max_itrs=10_0000, alpha=0.01, delta=0.5):
    xk = starting_point()
    iterations = []
    i = 0

    while i < max_itrs:
        iterations.append(xk)
        g = gradient(xk)

        if np.linalg.norm(g) < eps:
            break

        lambda_k = 300_000
        current_value = objective(xk)

        while objective(xk - lambda_k * g) > current_value - alpha * lambda_k * np.linalg.norm(g)**2:
            lambda_k *= delta

        xk = xk - lambda_k * g

        i += 1
    return xk, i, iterations

def gradient_optimal_step(step_bound = 300_000, eps=1e-6, max_itrs=100000):
    xk = starting_point()
    iterations = []
    i = 0

    while i < max_itrs:
        iterations.append(xk)
        g = gradient(xk)

        if np.linalg.norm(g) < eps:
            break

        phi = lambda l: objective(xk - l * g)

        result = minimize_scalar(
            phi,
            bounds=(0, step_bound),
            method="bounded"
        )

        lambda_k = result.x

        xk = xk - lambda_k * g


        i += 1

    return xk, i, iterations

In [ ]:
x_backtracking, it_backtracking, trajectory_backtracking = gradient_backtracking()

print("Gradientna metoda s backtrackingom")
print("Najdeny bod:", x_backtracking)
print("Pocet iteracii:", it_backtracking)
plot_slovakia(point=x_backtracking)

In [ ]:
x_optimal, it_optimal, trajectory_optimal = gradient_optimal_step()

print("Gradientna metoda s optimalnym krokom")
print("Najdeny bod:", x_optimal)
print("Pocet iteracii:", it_optimal)
plot_slovakia(point=x_optimal)


# Uloha g)



In [ ]:
def plot_slovakia_trajectories(point=None, trajectory1=None, trajectory2=None, trajectory3=None, show_cities=True):
    slovakia = gpd.read_file(
        "data/ne_10m_admin_0_countries.zip"
    )

    slovakia = slovakia[slovakia["ISO_A3"] == "SVK"]
    slovakia = slovakia.to_crs(epsg=5514)

    fig, ax = plt.subplots(figsize=(10,8))

    slovakia.plot(
        ax=ax,
        color='whitesmoke',
        edgecolor='black'
    )

    if show_cities:
        DF_MESTA.plot(
            ax=ax,
            color='red',
            markersize=np.sqrt(DF_MESTA['Population']) / 5,
            alpha=0.7
        )

        for x, y, city in zip(DF_MESTA.geometry.x, DF_MESTA.geometry.y, DF_MESTA['City']):
            ax.text(x, y, city, fontsize=8, ha='left', va='bottom')

    # trajectory
    if trajectory1 is not None:
        traj = np.array(trajectory1)
        ax.plot(traj[:, 0], traj[:, 1], marker='o', markersize=3, linewidth=1.5, color='blue', zorder=4)

    
    if trajectory2 is not None:
        traj = np.array(trajectory2)
        ax.plot(traj[:, 0], traj[:, 1], marker='o', markersize=3, linewidth=1.5, color='grey', zorder=4)


    if trajectory3 is not None:
        traj = np.array(trajectory3)
        ax.plot(traj[:, 0], traj[:, 1], marker='o', markersize=3, linewidth=1.5, color='red', zorder=4)


    #  final point
    if point is not None:
        ax.scatter(point[0], point[1], s=160, color="green", marker="x", zorder=5)


    minx, miny, maxx, maxy = slovakia.total_bounds

    cx = (minx + maxx) / 2
    cy = (miny + maxy) / 2

    sx = (maxx - minx) / 2
    sy = (maxy - miny) / 2

    zoom = 0.5  # 0.5 = 2x zoom (menší okno)

    ax.set_xlim(cx - sx * zoom, cx + sx * zoom)
    ax.set_ylim(cy - sy * zoom, cy + sy * zoom)

    ax.set_axis_off()
    plt.tight_layout()
    plt.show()



In [ ]:
plot_slovakia_trajectories(point=solution_d, trajectory1=trajectory_backtracking, trajectory2=trajectory_optimal, trajectory3=trajectory_constant, show_cities=True)
plot_slovakia_trajectories(point=None, trajectory1=trajectory_backtracking, show_cities=False)
plot_slovakia_trajectories(point=None, trajectory1=trajectory_optimal, show_cities=True)
plot_slovakia_trajectories(point=None, trajectory1=trajectory_constant, show_cities=False)

# Uloha h)

In [ ]:
# Uloha H - nacitanie rozsirenych dat
DF_MESTA_EXT = pd.read_csv(
    'data/slovensko_extended.csv',
    sep=','
)

M_EXT = len(DF_MESTA_EXT)
SUCET_POPULACIE_EXT = DF_MESTA_EXT['Population'].sum()
DF_MESTA_EXT['Weight'] = DF_MESTA_EXT['Population'] / SUCET_POPULACIE_EXT

DF_MESTA_EXT['geometry'] = [
    Point(xy) for xy in zip(DF_MESTA_EXT['Longitude'], DF_MESTA_EXT['Latitude'])
]

DF_MESTA_EXT = gpd.GeoDataFrame(
    DF_MESTA_EXT,
    geometry='geometry',
    crs='EPSG:4326'
)
DF_MESTA_EXT = DF_MESTA_EXT.to_crs(epsg=5514)

COORDS_EXT = np.column_stack([
    DF_MESTA_EXT.geometry.x,
    DF_MESTA_EXT.geometry.y,
])

WEIGHTS_EXT = DF_MESTA_EXT['Weight'].to_numpy()[:, None]

print(f'Nacitanych miest: {M_EXT}')
DF_MESTA_EXT.head()


In [ ]:
def plot_slovakia_extended(point=None):
    slovakia_ext = gpd.read_file(
        "data/ne_10m_admin_0_countries.zip"
    )

    slovakia_ext = slovakia_ext[slovakia_ext["ISO_A3"] == "SVK"]
    slovakia_ext = slovakia_ext.to_crs(epsg=5514)

    fig, ax = plt.subplots(figsize=(10, 8))

    slovakia_ext.plot(
        ax=ax,
        color='whitesmoke',
        edgecolor='black'
    )

    DF_MESTA_EXT.plot(
        ax=ax,
        color='red',
        markersize=np.sqrt(DF_MESTA_EXT['Population']) / 5,
        alpha=0.7
    )

    if point is not None:
        ax.scatter(
            point[0], point[1],
            s=160,
            color='green',
            marker='x',
            zorder=5
        )

    for x, y, city in zip(
        DF_MESTA_EXT.geometry.x,
        DF_MESTA_EXT.geometry.y,
        DF_MESTA_EXT['City']
    ):
        ax.text(x, y, city, fontsize=7, ha='left', va='bottom')

    ax.set_axis_off()
    plt.show()

plot_slovakia_extended()


## Cast D s rozsirenym datasetom

In [ ]:
WC_EXT = WEIGHTS_EXT * COORDS_EXT
EPS = 10**-6

x0_ext = np.sum(WC_EXT, axis=0)
xk_ext = x0_ext.copy()
i_ext = 0

while True:
    d_ext = np.linalg.norm(xk_ext - COORDS_EXT, axis=1)[:, None]

    # Ochrana proti deleniu nulou, ak by xk_ext presne padol do niektoreho mesta
    d_ext = np.where(d_ext == 0, 10**-12, d_ext)

    xkk_ext = np.sum(WC_EXT / d_ext, axis=0) / np.sum(WEIGHTS_EXT / d_ext)
    i_ext += 1

    if np.linalg.norm(xkk_ext - xk_ext) < EPS:
        xk_ext = xkk_ext
        break

    xk_ext = xkk_ext

print(f'Weiszfeld finished in {i_ext} iterations, found: {xk_ext}')
plot_slovakia_extended(xk_ext)

## Cast E s rozsirenym datasetom

In [ ]:
def gradient_basic_extended(step, eps, max_itrs):
    WC_EXT = WEIGHTS_EXT * COORDS_EXT
    x0_ext = np.sum(WC_EXT, axis=0)
    xk_ext = x0_ext.copy()

    i_ext = 0
    while True:
        d_ext = np.linalg.norm(xk_ext - COORDS_EXT, axis=1)[:, None]

        # Ochrana proti deleniu nulou
        d_ext = np.where(d_ext == 0, 10**-12, d_ext)

        g_ext = np.sum(WEIGHTS_EXT * (xk_ext - COORDS_EXT) / d_ext, axis=0)

        if np.linalg.norm(g_ext) < eps or i_ext > max_itrs:
            break

        xk_ext = xk_ext - g_ext * step
        i_ext += 1

    return xk_ext, i_ext

x_ext, i_ext = gradient_basic_extended(100_000, 10**-6, 100_000)
print(f'Gradient finished in {i_ext} iterations, found: {x_ext}')
plot_slovakia_extended(x_ext)

step = 10
while step < 10**7:
    x_ext, i_ext = gradient_basic_extended(step, 10**-6, 100_000)
    print(f'Step {step} finished in {i_ext} iterations, found: {x_ext}')
    step *= 2

# cast H s rozsirenym datasetom

In [ ]:
W_EXT = WEIGHTS_EXT.reshape(-1)

# ucelova funkcia
def objective_extended(x):
    distances = np.linalg.norm(COORDS_EXT - x, axis=1)
    return np.sum(W_EXT * distances)


# gradient ucelovej funkcie
def gradient_extended(x):
    distances = np.linalg.norm(x - COORDS_EXT, axis=1)

    distances = np.maximum(distances, 1e-12)

    g = np.sum(W_EXT[:, None] * (x - COORDS_EXT) / distances[:, None], axis=0)
    return g


# startovaci bod rovnaky ako v casti d) a e)
def starting_point_extended():
    return np.sum(W_EXT[:, None] * COORDS_EXT, axis=0)


# f1) gradientna metoda s backtrackingom
def gradient_backtracking_extended(eps=1e-6, max_itrs=10_0000, alpha=0.01, delta=0.5):
    xk = starting_point_extended()
    i = 0

    while i < max_itrs:
        g = gradient_extended(xk)

        if np.linalg.norm(g) < eps:
            break

        lambda_k = 300_000
        current_value = objective_extended(xk)

        while objective_extended(xk - lambda_k * g) > current_value - alpha * lambda_k * np.linalg.norm(g)**2:
            lambda_k *= delta

        xk = xk - lambda_k * g

        i += 1
    return xk, i

def gradient_optimal_step_extended(step_bound = 300_000, eps=1e-6, max_itrs=100000):
    xk = starting_point_extended()
    i = 0

    while i < max_itrs:
        g = gradient_extended(xk)

        if np.linalg.norm(g) < eps:
            break

        phi = lambda l: objective_extended(xk - l * g)

        result = minimize_scalar(
            phi,
            bounds=(0, step_bound),
            method="bounded"
        )

        lambda_k = result.x

        xk = xk - lambda_k * g


        i += 1

    return xk, i

In [ ]:
x_backtracking_ext, it_backtracking_ext = gradient_backtracking_extended()

print("Gradientna metoda s backtrackingom")
print("Najdeny bod:", x_backtracking_ext)
print("Pocet iteracii:", it_backtracking_ext)
plot_slovakia_extended(point=x_backtracking_ext)

In [ ]:
x_optimal_ext, it_optimal_ext = gradient_optimal_step_extended()

print("Gradientna metoda s optimalnym krokom")
print("Najdeny bod:", x_optimal_ext)
print("Pocet iteracii:", it_optimal_ext)
plot_slovakia_extended(point=x_optimal_ext)

# Uloha h2)

## Lokalizacny problem s viacerymi recyklacnymi stanicami

Majme rozsirenie povodnej ulohy, kde namiesto polohy pre jednu recyklacnu stanicu hladame optimalnu polohu pre $n$ recyklacnych stanic. Funkciu, ktoru optimalizujeme je v podstate rovnaka ako v povodnom zadani, len s tym rozdielom, ze kazde mesto si ako svoju recyklacnu stanicu zvoli tu najblizsiu k sebe. Matematicky sa tato uloha da naformulovat:

$$
\min_{x_1, \dots, x_n} F(x_1, \dots, x_n) = \sum_{i=1}^m w_i \min_{j=1,\dots,n} \|a^i - x_j \|_2
$$

kde $n$ je pocet recyklacnych stanic, $m$ je pocet miest,  $x_1, \dots, x_n$ su polohy recyklacnych stanic. Predpokladajme, ze $n \le m$, inak je riesenie trvivialne.

Ak uz mame priradene mesta k recyklacnym staniciam, ulohu vieme preformulovat nasledovne:
$$
\min_{x_1, \dots, x_n} F(x_1, \dots, x_n) = \sum_{j=1}^n \sum_{i\in C_j} w_i \| a^i - x_j \|_2
$$

kde $C_j \quad j \in \{1,\dots, n\}$ su mnoziny s mestami - t.j. mesta, ktore prisluchaju $j$ recyklacnej stanici.

Ked mame takyto problem, dokazeme ho riesit rovnako ako v ulohe c), len s tym rozdielom, ze ten algoritmus spustime n-krat pre kazdu recyklacnu stanicu. 


Celu ulohu hladania polohy recyklacnych stanic budeme riesit nasledovnym iterativnym sposobom:
1. Na zaciatku nahodne priradime mesta k recyklacnym staniciam. (prvych n hodnot priradenia bude z nahodnej permutacie n cisiel, aby kazda stanica na zaciatku mala aspon jedno pridelene mesto )
2. Vypocitaj optimalne pozicie stanic vzhladom na momentalne priradenie
3. Pre kazde mesto zisti, ku ktorej stanici je najblizsie a takymto sposobom zisti nove priradenia
4. Ak existuju recyklacne stanice bez mesta, zober mesto od recyklacnej stanice s viacerymi mestami a prirad toto mesto prazdnej stanici
5. Ak sa priradenia nezmenili, skonci algoritmus, inak pokracuj krokom 2.

Tento algoritmus konverguje k lokalnemu minimu, pretoze v kazdom kroku robime operacie, ktore bud nezmenia alebo znizia hodnotu ucelovej funkcie a tato funkcia je zdola ohranicena nulou. Aby sme mali zaruku, ze sme nasli globalne minimum, museli by sme vyskusat vsetkych $n!$ moznosti priradenia, co je pre pocet miest $\ge 32$ vypoctovo neuskutocnitelne. Avsak ak sa chceme co najviac priblizit ku globalnemu optimu, mozeme algoritmus spustit viackrat z inych zaciatocnych miest.

_Poznamka_: V 1. kroku sa da vybrat rozdelenie heuristicky lepsie tak, ze stanice maju pociatocnu polohu v mestach. Prve mesto - polohu prvej stanice vyberieme nahodne a nasledujuce mesta vyberame tak, ze mesta vzdialenejsie od uz vybratych miest budu mat vacsiu pravdepodobnost, ze budu zvolene. Opakujeme, dokym sme nevybrali $n$ miest. Z tohto nasledne vytvorime prvotne rozdelenie.



In [ ]:
def weiszfeld(coords:np.ndarray, weights:np.ndarray, epsilon=10**-6):
    assert len(coords) > 0

    if len(coords) == 1:
        return coords[0]
    
    wc = weights * coords

    x0 = np.sum(wc, axis=0)
    xk = x0.copy()
    i = 0

    while True:
        d = np.linalg.norm(xk-coords, axis=1)[:, None]

        xkk = np.sum(wc/d, axis=0) / np.sum(weights / d)
        i += 1

        if np.linalg.norm(xkk - xk) < epsilon:
            xk = xkk
            break

        xk = xkk
    
    # print(f'Weiszfeld finished in {i} iterations')
    
    return xk


def find_optimal_locations(n:int, m:int, coords:np.ndarray, weights:np.ndarray, seed=420, show_steps=True):
    rng = np.random.default_rng(seed)

    start = rng.permutation(n)
    end = rng.integers(0, n, size=m-n)
    # pole o dlzky m, kde na i-tej pozicii je hodnota 0 az (n-1) urcujuca ku ktorej stanici i-te mesto patri
    assignments =  np.concatenate([start,end])
    i = 0

    while True:
        # Step 2
        optimal_locations = np.empty((n,2))
        for i in range(n):
            mask = assignments == i
            optimal_locations[i] = weiszfeld(coords[mask], weights[mask])
        
        # Step 3
        old_assignments = assignments.copy()

        for i in range(m):
            assignments[i] = np.argmin(np.linalg.norm(coords[i] - optimal_locations, axis=1))

        # Step 4
        counts = np.bincount(assignments, minlength=n)
        j = 0
        for i in range(n):
            if counts[i] != 0:
                continue

            print('switched')
            
            while counts[j] < 2:
                j += 1
            
            k = np.where(assignments == j)[0][0]
            assignments[k] = i
            counts[j] -= 1
            counts[i] += 1

        # print(counts)
        # print(assignments)
        # print(np.bincount(assignments))

        
        if np.array_equal(old_assignments, assignments):
            if show_steps:
                print(f'Location assignments converged in {i} iterations')
            return assignments, optimal_locations
        
        i += 1
    
        
        
def objective_function_value(station_locations, city_coords, weights):
    return np.sum( weights * np.min(np.linalg.norm(station_locations[:, None, :] - city_coords[None, :, :], axis=2), axis=0)[:,None] )



ITRS = 5
for x in [2, 3, 5, 10]:
    best = float('inf')
    b_assign = None
    b_points = None
    for i in range(ITRS):
        rseed = random.randint(0,100_000)
        print('SEED', rseed)
        assign, points = find_optimal_locations(x, M, COORDS, WEIGHTS, rseed, show_steps=False)
        val = objective_function_value(points, COORDS, WEIGHTS)
        if val < best:
            best, b_assign, b_points = val, assign, points
    
    print(f'After {ITRS} random starts found {x} locations with function value: {best}')
    plot_slovakia(b_points, b_assign, cross_color='red')
